In [ ]:
# Lock Kaggle's loaded core C-libraries, then install scanpy safely
!pip freeze | grep -E "numpy|scipy|pandas" > constraints.txt
!pip install scanpy -c constraints.txt -q

In [ ]:
import scanpy as sc
import pandas as pd
import os

sc.settings.verbosity = 2

# Primary path
h5ad_path = "/kaggle/input/datasets/ericdu847/microglial-annotated-multiregion-nuclei-data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad"

# Fallback path check
if not os.path.exists(h5ad_path):
    fallback_paths = [
        "/kaggle/input/microglial-annotated-multiregion-nuclei-data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad",
        "/kaggle/input/Microglial annotated multiregion nuclei data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad"
    ]
    for alt in fallback_paths:
        if os.path.exists(alt):
            h5ad_path = alt
            break

print(f"Loading dataset from: {h5ad_path}")
adata = sc.read_h5ad(h5ad_path)
print("Data loaded successfully. Shape:", adata.shape)

In [ ]:
# 1. Identify the Donor ID column dynamically using robust substring matching
donor_col = None
donor_keywords = ['donor', 'subject', 'individual', 'patient']

for col in adata.obs.columns:
    if any(keyword in col.lower() for keyword in donor_keywords):
        donor_col = col
        break

# If auto-detect fails or guesses wrong, catch it cleanly
if donor_col is None or donor_col not in adata.obs.columns:
    print("Available columns in metadata:")
    print(list(adata.obs.columns))
    raise KeyError("Could not automatically detect the donor/patient ID column. Please look at the printed list above and manually set 'donor_col' to the exact name.")

print(f"Using '{donor_col}' to identify unique patients.")

# 2. Extract unique patients
# adata.obs contains cell-level data; dropping duplicates on the donor column leaves patient-level data
patient_metadata = adata.obs.drop_duplicates(subset=[donor_col]).copy()

# 3. Clean up the dataframe (Optional but recommended)
# This removes technical, cell-specific columns that don't make sense on a patient-level summary table.
cols_to_drop = [c for c in patient_metadata.columns if any(k in c.lower() for k in ['cluster', 'barcode', 'umap', 'n_genes', 'leiden', 'cell_type', 'region', 'roi'])]
patient_metadata = patient_metadata.drop(columns=cols_to_drop, errors='ignore')

# 4. Save to CSV
output_csv = "SEA_AD_Unique_Patient_Metadata.csv"
patient_metadata.to_csv(output_csv, index=False)

print(f"\nExtraction Complete!")
print(f"Total Unique Patients: {len(patient_metadata)}")
print(f"Saved patient data to: {output_csv}")

# Display the first few rows for a quick sanity check
display(patient_metadata.head())